# Linear Regression

The idea is to explore a classifier named Logistic Regression.
However, to introduce the inner workings of such a model, let us start to analyse the Linear Regression one.

This will allow us to understand:
1. what is a linear model
2. define a cost function to train a model
3. how to fit a linear model given a cost function
4. Traditional optimization methods

The linear regression models is defined by:
$$
h(x,m,b) = m\times x + b
$$

The typical cost function to fit a linear regression is the following:
$$
e = \frac{\sum_{i=0}^{n}(y_i-h(x_i, m, b))^2}{2n}
$$

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pyBlindOpt.init as init
import pyBlindOpt.pso as pso
import tqdm
from sklearn.datasets import make_regression

plt.rcParams["figure.figsize"] = [8, 5]
plt.rcParams["figure.dpi"] = 100

In [ ]:
def rmse(predictions, targets):
    return np.sqrt(((predictions - targets) ** 2).mean())

In [ ]:
x, y = make_regression(n_samples=1000, n_features=1, noise=7, random_state=42)

In [ ]:
# plot the data
plt.plot(x, y, "o")
plt.show()

In [ ]:
class LRPSO:
    def _f(self, X, w):
        wt = w[np.newaxis].T
        y_hat = wt[0] + np.dot(X, wt[1:])
        return y_hat

    def _cost(self, X, y, w):
        y_hat = self._f(X, w)
        return np.mean(np.power((y - y_hat.flatten()), 2))

    def fit(self, X, y, n_pop=30, iter=100, verbose=False):
        bounds = np.asarray([(-50.0, 50.0), (-50.0, 50.0)])
        population = init.oblesa(lambda w: self._cost(X, y, w), bounds, n_pop=n_pop)
        solution = pso.particle_swarm_optimization(
            lambda w: self._cost(X, y, w), bounds, population=population, n_iter=iter, verbose=verbose, debug=True
        )
        self.w = solution[0]
        return solution[2][0]

    def predict(self, X):
        wt = self.w[np.newaxis].T
        y_hat = wt[0] + np.dot(X, wt[1:])
        return y_hat.flatten()

    def params(self):
        return self.w

In [ ]:
lr = LRPSO()
objs = lr.fit(x, y)
print(f"LR {lr.params()}")

In [ ]:
plt.plot(range(0, 100), objs)
plt.show()

In [ ]:
y_hat = lr.predict(x)
# plot the data
plt.plot(x, y, "o")
plt.plot(x, y_hat, "ro")
plt.show()

In [ ]:
cost = rmse(y, y_hat)
print(f"RMSE = {cost}")

In [ ]:
class LR:
    def fit(self, X, y, lr=0.01, iter=1000):
        def _f(w):
            wt = w[np.newaxis].T
            y_hat = wt[0] + np.dot(X, wt[1:])
            return y_hat

        nd = 2 if len(X.shape) == 1 else (X.shape[1] + 1)
        self.w = np.array([1.0] * nd)
        for _ in tqdm.tqdm(range(iter)):
            y_hat = _f(self.w)  # The current predicted value of Y
            diff = y - y_hat.flatten()
            D_m = []
            for i in range(X.shape[1]):
                D_m.append(np.average(-X[:, i] * diff))  # Derivative wrt m
            D_m = np.array(D_m)
            D_c = -np.average(diff)  # Derivative wrt c
            self.w[1:] = self.w[1:] - lr * D_m  # Update m
            self.w[0] = self.w[0] - lr * D_c  # Update c

    def predict(self, X):
        wt = self.w[np.newaxis].T
        y_hat = wt[0] + np.dot(X, wt[1:])
        return y_hat.flatten()

    def params(self):
        return self.w

In [ ]:
lr = LR()
lr.fit(x, y)
print(f"LR {lr.params()}")

In [ ]:
y_hat = lr.predict(x)
# plot the data
plt.plot(x, y, "o")
plt.plot(x, y_hat, "ro")
plt.show()

In [ ]:
cost = rmse(y, y_hat)
print(f"RMSE = {cost}")

In [ ]:
class LRAG:
    def fit(self, X, y, lr=0.01, iter=300):
        def _J(w):
            wt = w[jnp.newaxis].T
            y_hat = wt[0] + jnp.dot(X, wt[1:])
            return jnp.mean((y - y_hat.flatten()) ** 2.0) + 1e-5 * (jnp.sum(w)) ** 2

        g = jax.grad(_J)
        nd = 2 if len(X.shape) == 1 else (X.shape[1] + 1)
        w = np.array([1.0] * nd)

        for _ in tqdm.tqdm(range(iter)):
            w = w - lr * g(w)

        self.w = w

    def predict(self, X):
        wt = self.w[np.newaxis].T
        y_hat = wt[0] + np.dot(X, wt[1:])
        return y_hat.flatten()

    def params(self):
        return self.w

In [ ]:
lr = LRAG()
lr.fit(x, y)
print(f"LR {lr.params()}")

In [ ]:
y_hat = lr.predict(x)
# plot the data
plt.plot(x, y, "o")
plt.plot(x, y_hat, "ro")
plt.show()

In [ ]:
cost = rmse(y, y_hat)
print(f"RMSE = {cost}")

In [ ]:
class LRAGNewton:
    def fit(self, X, y, maxiter=100, tol=1e-5):
        def _J(w):
            wt = w[jnp.newaxis].T
            y_hat = wt[0] + jnp.dot(X, wt[1:])
            return jnp.mean((y - y_hat.flatten()) ** 2.0) + 1e-5 * (jnp.sum(w)) ** 2

        g = jax.grad(_J)
        h = jax.hessian(_J)
        nd = 2 if len(X.shape) == 1 else (X.shape[1] + 1)
        w = np.array([1.0] * nd)

        for _ in tqdm.tqdm(range(maxiter)):
            delta = np.linalg.solve(h(w), -g(w))
            w = w + delta
            if np.linalg.norm(delta) < tol:
                break

        self.w = w

    def predict(self, X):
        wt = self.w[np.newaxis].T
        y_hat = wt[0] + np.dot(X, wt[1:])
        return y_hat.flatten()

    def params(self):
        return self.w

In [ ]:
lr = LRAGNewton()
lr.fit(x, y)
print(f"LR {lr.params()}")

In [ ]:
y_hat = lr.predict(x)
# plot the data
plt.plot(x, y, "o")
plt.plot(x, y_hat, "ro")
plt.show()

In [ ]:
cost = rmse(y, y_hat)
print(f"RMSE = {cost}")

## Non Linear Data

In [ ]:
x, y = make_regression(n_samples=1000, n_features=1, noise=7, random_state=42)
x = x[:, 0]
y = np.square(y)

# plot the data
plt.plot(x, y, "o")
plt.show()

In [ ]:
x_squared = np.square(x)
X = np.column_stack((x, x_squared))

In [ ]:
lr = LRAGNewton()
lr.fit(X, y)
print(f"LR {lr.params()}")

In [ ]:
y_hat = lr.predict(X)

# plot the data
plt.plot(x, y, "o")
plt.plot(x, y_hat, "ro")
plt.show()

## Summary: four solvers, one answer

The closed-form least-squares solution (`notebook_01`) is the reference. Every optimiser must land on (almost) the same line.
The differences are *cost* and *requirements*: PSO needs nothing but the loss, gradient descent needs the gradient, Newton needs the
gradient and the Hessian, and the closed form needs the model to be linear.

In [ ]:
x, y = make_regression(n_samples=1000, n_features=1, noise=7, random_state=42)
w_exact = np.linalg.lstsq(np.c_[np.ones(len(x)), x], y, rcond=None)[0]
print(f"{'solver':22}{'intercept':>10}{'slope':>9}{'RMSE':>8}")
print(f"{'closed form (lstsq)':22}{w_exact[0]:10.3f}{w_exact[1]:9.3f}{rmse(y, w_exact[0] + x[:, 0] * w_exact[1]):8.3f}")
for name, solver in {"PSO": LRPSO(), "gradient descent": LR(), "autodiff GD": LRAG(), "Newton": LRAGNewton()}.items():
    solver.fit(x, y)
    w = solver.params()
    print(f"{name:22}{w[0]:10.3f}{w[1]:9.3f}{rmse(y, solver.predict(x)):8.3f}")

## Exercises

1. Reduce the number of iterations of gradient descent to 20. Which solvers are still accurate?
2. Replace the squared error by the absolute error in `LRPSO`. Can gradient descent still be used? Can PSO?
3. Add 20 outliers (`y += 500` for 20 points) and refit. Which solver is affected most? (Robustness of the loss matters for poisoning: `01-spam/notebook_07`.)